# SkyEye — 天气分类
## EfficientNet-B4 → 知识蒸馏 → B0 → 结构化剪枝 → ONNX 导出

按顺序执行以下 Cell 即可完成完整训练管线。

> 依赖已预装在 `.venv/` 中，激活: `.venv\\Scripts\\activate`

## 1. 环境检查

检查 Python、PyTorch、CUDA 版本及 GPU 显存信息。

In [ ]:
import sys, os
print(f"Python: {sys.version}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

print("\n" + "=" * 60)
print("✓ 环境检查通过")
print("=" * 60)

## 2. 数据准备 + DataLoader

自动发现 `datasets/` 下所有数据源，合并到 `_data/weather`（首次运行需要，后续自动跳过）。按类分布、划分训练/验证集，构建 DataLoader。

In [ ]:
import numpy as np, torch
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from sklearn.model_selection import train_test_split
from config import CONFIG
from data.dataset import prepare_data
from data.augmentations import get_train_transforms, get_val_transforms

data_root = prepare_data()

full_dataset = ImageFolder(data_root)
num_classes = len(full_dataset.classes)
class_counts = np.bincount(full_dataset.targets, minlength=num_classes)

indices = np.arange(len(full_dataset))
train_idx, val_idx = train_test_split(
    indices, test_size=CONFIG["val_split"],
    stratify=full_dataset.targets, random_state=CONFIG["seed"],
)

train_ds = torch.utils.data.Subset(
    ImageFolder(data_root, transform=get_train_transforms(CONFIG["img_size"])), train_idx)
val_ds = torch.utils.data.Subset(
    ImageFolder(data_root, transform=get_val_transforms(CONFIG["img_size"])), val_idx)

nw, pin = CONFIG["num_workers"], torch.cuda.is_available()
train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,
                          num_workers=nw, pin_memory=pin)
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False,
                        num_workers=nw, pin_memory=pin)

print(f"Device: {CONFIG['device']} | Classes: {full_dataset.classes}")
print(f"Distribution: {dict(zip(full_dataset.classes, class_counts.astype(int)))}")
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")
print(f"\n{'=' * 60}")
print("✓ 数据准备完成")
print("=" * 60)

## 3. 训练教师模型 (EfficientNet-B4)

训练 EfficientNet-B4 教师模型，输出 `results/teacher_best.pth`。具体超参数见 `config.py`。

In [ ]:
from training.train_teacher import train_teacher
teacher = train_teacher()
print("\n" + "=" * 60)
print("✓ 教师模型训练完成")
print("=" * 60)

## 4. 知识蒸馏 (B4 → B0)

用教师模型（B4）的知识蒸馏训练学生模型（B0），输出 `results/student_distilled_best.pth`。

In [ ]:
from training.distill_student import run_distillation
student = run_distillation()
print("\n" + "=" * 60)
print("✓ 知识蒸馏完成")
print("=" * 60)

## 5. 结构化剪枝 + 渐进微调

对蒸馏后的 B0 学生模型进行结构化剪枝，然后渐进微调恢复精度，输出 `results/student_pruned_final.pth`。

In [ ]:
from training.prune_finetune import prune_and_finetune
pruned_model = prune_and_finetune()
print("\n" + "=" * 60)
print("✓ 剪枝微调完成")
print("=" * 60)

## 6. ONNX 导出 + INT8 量化 + CPU 测速

将剪枝后的模型导出为 ONNX 格式，再进行 INT8 量化，最后在 CPU 上测速验证推理耗时。

输出：
- `results/weather_model.onnx` — FP32 ONNX 模型
- `results/weather_model_int8.onnx` — INT8 量化模型

In [ ]:
from inference.export_onnx import export_to_onnx, quantize_to_int8, benchmark_cpu
onnx_path = export_to_onnx('results/student_pruned_final.pth')
int8_path = quantize_to_int8(onnx_path)
benchmark_cpu(onnx_path, int8_path)
print("\n" + "=" * 60)
print("✓ ONNX 导出 + 量化 + 测速完成")
print("=" * 60)

## 7. (可选) CPU 单张推理测试

用导出的模型对单张图片进行推理验证。

In [ ]:
# from inference.infer import predict_image
# result = predict_image('your_image.jpg')
# print(f'Prediction: {result["prediction"]} (Confidence: {result["confidence"]:.4f})')
# for name, prob in result['top_k']:
#     print(f'  {name}: {prob:.4f}')
print("\n" + "=" * 60)
print("✓ 取消注释并替换图片路径即可推理")
print("=" * 60)